# KVO_05 — SAM 3 zero-shot tropical crown segmentation benchmark

**Question:** can Meta SAM 3 delineate individual tropical tree crowns from UAV RGB imagery without task-specific training?

This notebook deliberately evaluates **segmentation only**. It does not use the Musanga U-Net probabilities and does not ask SAM 3 to classify species.

Primary experiment: for a reproducible stratified sample of manual crowns ≥5 m², crop RGB context around each crown and give SAM 3 a **visual box prompt**. Compare the returned zero-shot mask with the independently digitized crown polygon using IoU, Dice, area bias and centroid displacement. The box is expanded beyond the manual crown bounds so the prompt does not simply reveal the exact boundary.

Why box rather than point in v1? The official SAM 3 image API explicitly documents visual box prompting, while point prompting is described for interactive refinement. A later extension can test point-only prompts once the benchmark is operational.

**Important:** the manual crown polygon is used only to construct the prompt and reference mask; no SAM fine-tuning occurs. Results therefore measure prompt-conditioned crown delineation, not automatic crown discovery.

Official SAM 3 requires Python ≥3.12, PyTorch ≥2.7 and a CUDA-capable GPU. Model checkpoint access must first be accepted on Meta's Hugging Face repository.


## 0 — Runtime and installation

Use a **GPU runtime**. The official SAM 3 repository is installed directly; no third-party wrapper is used.

If Hugging Face checkpoint access has not yet been granted, request it at the official `facebook/sam3` model page and authenticate in Colab. The token is requested interactively and is not stored in this notebook.


In [ ]:
import sys, subprocess, os
print("Python:",sys.version)
assert sys.version_info >= (3,12), "SAM 3 requires Python >=3.12. Use a current Colab runtime."
subprocess.run(["nvidia-smi"],check=False)


In [ ]:
!pip -q install -U huggingface_hub geopandas rasterio pyogrio shapely scikit-learn matplotlib tqdm
!pip -q install "git+https://github.com/facebookresearch/sam3.git"


In [ ]:
from huggingface_hub import login
# Run once per fresh runtime if the checkpoint is not already accessible.
# Your Hugging Face account must have accepted Meta's SAM 3 terms first.
login()


## 1 — Imports, data and benchmark settings

In [ ]:
from pathlib import Path
import json, hashlib, random, time, warnings, gc
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
import rasterio
from rasterio.features import rasterize, shapes
from rasterio.windows import Window
from rasterio.transform import xy
from shapely.geometry import shape
from tqdm.auto import tqdm
import torch
from PIL import Image
from google.colab import drive
warnings.filterwarnings("ignore")

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "GPU strongly recommended/expected for SAM 3."
DEVICE="cuda"
print(torch.cuda.get_device_name(0))


In [ ]:
DRIVE_ROOT=Path("/content/drive"); MYDRIVE=DRIVE_ROOT/"MyDrive"
def healthy():
    try: return MYDRIVE.exists() and any(MYDRIVE.iterdir())
    except OSError: return False
if not healthy(): drive.mount(str(DRIVE_ROOT),force_remount=True)

ROOT=MYDRIVE/"Colab Notebooks"/"Musanga"; GEO=ROOT/"data"/"Geo_Ref"; AD=GEO/"Annotations"
OUT=ROOT/"KVO_work"/"KVO_05_SAM3_crown_benchmark"; OUT.mkdir(parents=True,exist_ok=True)
VIS=OUT/"examples"; VIS.mkdir(exist_ok=True)

SITES={
 "MOS_1_2":{"raster":GEO/"MOS_1_2_kvo.tif","annotations":AD/"Tree_Annotations_MOS_1_2_kvo.shp"},
 "Yoko":{"raster":GEO/"Yoko_kvo_georef.tif","annotations":AD/"Tree_Annotations_Yoko_kvo_georef.shp"},
 "Baego":{"raster":GEO/"Baego_georef.tif","annotations":AD/"Tree_Annotation_Baego.shp"},
 "Yangambi":{"raster":GEO/"Yangambi_georef.tif","annotations":AD/"Tree_Annotation_Yangambi_georef.shp"}}
for s,d in SITES.items(): assert d["raster"].exists() and d["annotations"].exists(),s

MIN_AREA_M2=5.0
N_PER_SITE=200
CROP_MARGIN_FRAC=0.75     # context around crown bbox
BOX_EXPAND_FRAC=0.15      # SAM prompt box is 15% larger than manual bbox
MIN_CROP_PX=384
FORCE_RUN=False
print("OUT:",OUT)


## 2 — Audit crowns and make a reproducible size-stratified sample

Sampling is independent of Musanga class because SAM is being tested as a generic crown segmenter. Within each site we stratify by crown-area quartile so tiny/medium/large eligible crowns are represented.


In [ ]:
ANNS={}; samples=[]
for site,d in SITES.items():
    with rasterio.open(d["raster"]) as src: crs=src.crs
    g=gpd.read_file(d["annotations"])
    if g.crs!=crs: g=g.to_crs(crs)
    g=g[g.geometry.notna() & ~g.geometry.is_empty].copy()
    g["class_id_num"]=pd.to_numeric(g["Class_ID"],errors="coerce")
    g=g[g.class_id_num.notna()].copy(); g["class_id_num"]=g.class_id_num.astype(int)
    g["target"]=(g.class_id_num==1).astype(np.uint8)
    g["crown_area_m2"]=g.geometry.area.astype(float)
    g=g[g.crown_area_m2>=MIN_AREA_M2].copy().reset_index(drop=True)
    g["crown_id"]=np.arange(len(g),dtype=np.int64)
    g["area_bin"]=pd.qcut(g.crown_area_m2,4,labels=False,duplicates="drop")
    ANNS[site]=g
    take=[]
    rng=np.random.default_rng(SEED+list(SITES).index(site))
    for b,a in g.groupby("area_bin"):
        n=min(len(a),int(np.ceil(N_PER_SITE/a.area_bin.nunique())))
        take.extend(rng.choice(a.index,size=n,replace=False).tolist())
    sm=g.loc[take].copy()
    if len(sm)>N_PER_SITE: sm=sm.sample(N_PER_SITE,random_state=SEED)
    sm["site"]=site; samples.append(sm)
    print(site,"eligible",len(g),"sample",len(sm),"area median",round(g.crown_area_m2.median(),2))
sample=pd.concat(samples,ignore_index=True)
sample[["site","crown_id","class_id_num","target","crown_area_m2","area_bin"]].to_csv(OUT/"benchmark_sample.csv",index=False)
display(sample.groupby(["site","area_bin"]).size().unstack(fill_value=0))


## 3 — Load official SAM 3

`build_sam3_image_model()` downloads/loads the authorized official checkpoint through the SAM 3 package/Hugging Face setup. This can take time on the first run.


In [ ]:
print("Loading official SAM 3...")
sam3_model=build_sam3_image_model()
sam3_processor=Sam3Processor(sam3_model)
print("SAM 3 ready.")


## 4 — Crown crop + expanded visual-box prompt

The crop contains substantial surrounding canopy. The prompt box is deliberately expanded by 15% relative to the manual crown bounding box. This prevents the benchmark from reducing to tracing the exact reference rectangle.

SAM 3 can return multiple masks. We select the highest-scoring mask associated with the prompted crown. If the installed API exposes multiple visual-prompt outputs differently, the helper raises a clear message rather than silently guessing.


In [ ]:
def crop_for_crown(src,geom):
    w=rasterio.windows.from_bounds(*geom.bounds,transform=src.transform).round_offsets().round_lengths()
    bw,bh=max(1,int(w.width)),max(1,int(w.height))
    mx=max(int(bw*CROP_MARGIN_FRAC),(MIN_CROP_PX-bw)//2,32)
    my=max(int(bh*CROP_MARGIN_FRAC),(MIN_CROP_PX-bh)//2,32)
    ew=Window(int(w.col_off)-mx,int(w.row_off)-my,bw+2*mx,bh+2*my)
    arr=src.read([1,2,3],window=ew,boundless=True,fill_value=0)
    rgb=np.moveaxis(arr,0,-1)
    tr=src.window_transform(ew)
    ref=rasterize([(geom,1)],out_shape=rgb.shape[:2],transform=tr,fill=0,dtype="uint8").astype(bool)
    # reference bbox expressed in crop pixel coordinates
    rr,cc=np.where(ref)
    x0,x1=float(cc.min()),float(cc.max()+1); y0,y1=float(rr.min()),float(rr.max()+1)
    dx=(x1-x0)*BOX_EXPAND_FRAC; dy=(y1-y0)*BOX_EXPAND_FRAC
    box=np.array([max(0,x0-dx),max(0,y0-dy),min(rgb.shape[1],x1+dx),min(rgb.shape[0],y1+dy)],np.float32)
    return rgb,ref,box,tr

def sam3_box_predict(rgb,box):
    state=sam3_processor.set_image(Image.fromarray(rgb))
    # Official SAM3 image processor supports geometric visual box prompts.
    # API names have changed across SAM3 releases; support the current documented variants.
    candidates=[]
    if hasattr(sam3_processor,"set_box_prompt"):
        out=sam3_processor.set_box_prompt(state=state, box=box.tolist())
    elif hasattr(sam3_processor,"add_box_prompt"):
        out=sam3_processor.add_box_prompt(state=state, box=box.tolist())
    elif hasattr(sam3_processor,"set_visual_prompt"):
        out=sam3_processor.set_visual_prompt(state=state, boxes=[box.tolist()])
    else:
        raise RuntimeError("Installed SAM3 processor exposes no recognized box-prompt method. Inspect the official example notebook/API before continuing.")
    masks=out["masks"]; scores=out.get("scores",None)
    if torch.is_tensor(masks): masks=masks.detach().float().cpu().numpy()
    masks=np.asarray(masks)
    while masks.ndim>3 and masks.shape[0]==1: masks=masks[0]
    if masks.ndim==2: masks=masks[None]
    if masks.ndim==4 and masks.shape[1]==1: masks=masks[:,0]
    if scores is None: j=0
    else:
        if torch.is_tensor(scores): scores=scores.detach().float().cpu().numpy()
        j=int(np.asarray(scores).reshape(-1).argmax())
    return np.asarray(masks[j]>0.5,dtype=bool), (float(np.asarray(scores).reshape(-1)[j]) if scores is not None else np.nan)

def metrics(ref,pred,pixel_area):
    inter=np.logical_and(ref,pred).sum(); union=np.logical_or(ref,pred).sum()
    nr=ref.sum(); npred=pred.sum()
    iou=inter/union if union else np.nan
    dice=2*inter/(nr+npred) if nr+npred else np.nan
    area_ref=nr*pixel_area; area_pred=npred*pixel_area
    rb=np.argwhere(ref); pb=np.argwhere(pred)
    cent=np.nan
    if len(rb) and len(pb):
        d=rb.mean(0)-pb.mean(0); cent=float(np.sqrt((d*d).sum())*np.sqrt(pixel_area))
    return iou,dice,area_ref,area_pred,(area_pred-area_ref)/area_ref if area_ref else np.nan,cent


## 5 — Run benchmark, resume safely

Results are appended crown-by-crown. A disconnected Colab session resumes from the existing CSV rather than rerunning completed crowns.


In [ ]:
RESULTS=OUT/"sam3_crown_metrics.csv"
done=set()
if RESULTS.exists() and not FORCE_RUN:
    old=pd.read_csv(RESULTS); done=set(zip(old.site,old.crown_id.astype(int)))
    print("Resuming; completed crowns:",len(done))
else:
    old=pd.DataFrame()

rows=[]
for site in SITES:
    ss=sample[sample.site==site]
    with rasterio.open(SITES[site]["raster"]) as src:
        pixarea=abs(src.transform.a*src.transform.e)
        for row in tqdm(ss.itertuples(),total=len(ss),desc=f"SAM3 {site}"):
            key=(site,int(row.crown_id))
            if key in done: continue
            rgb,ref,box,tr=crop_for_crown(src,row.geometry)
            try:
                pred,score=sam3_box_predict(rgb,box)
                if pred.shape!=ref.shape:
                    from PIL import Image as PILImage
                    pred=np.array(PILImage.fromarray(pred.astype(np.uint8)).resize((ref.shape[1],ref.shape[0]),resample=PILImage.Resampling.NEAREST)).astype(bool)
                iou,dice,ar,ap,bias,cd=metrics(ref,pred,pixarea)
                rec={"site":site,"crown_id":int(row.crown_id),"class_id":int(row.class_id_num),"truth":int(row.target),
                     "crown_area_m2":float(row.crown_area_m2),"area_bin":int(row.area_bin),
                     "sam_score":score,"iou":iou,"dice":dice,"reference_area_m2":ar,"sam_area_m2":ap,
                     "relative_area_bias":bias,"centroid_displacement_m":cd,"error":""}
            except Exception as e:
                rec={"site":site,"crown_id":int(row.crown_id),"class_id":int(row.class_id_num),"truth":int(row.target),
                     "crown_area_m2":float(row.crown_area_m2),"area_bin":int(row.area_bin),"error":repr(e)}
            rows.append(rec)
            # persist every 10 crowns
            if len(rows)%10==0:
                pd.concat([old,pd.DataFrame(rows)],ignore_index=True).to_csv(RESULTS,index=False)
                gc.collect(); torch.cuda.empty_cache()
res=pd.concat([old,pd.DataFrame(rows)],ignore_index=True)
res.to_csv(RESULTS,index=False)
print("Saved:",RESULTS,"rows:",len(res))


## 6 — Primary results by site and crown size

In [ ]:
ok=res[res.error.fillna("")==""].copy()
summary=ok.groupby("site").agg(n=("iou","size"),median_iou=("iou","median"),mean_iou=("iou","mean"),
    median_dice=("dice","median"),mean_dice=("dice","mean"),median_abs_area_bias=("relative_area_bias",lambda x: np.median(np.abs(x))),
    median_centroid_displacement_m=("centroid_displacement_m","median")).reset_index()
display(summary); summary.to_csv(OUT/"sam3_site_summary.csv",index=False)

fig,ax=plt.subplots(figsize=(7,5))
for site,d in ok.groupby("site"):
    ax.scatter(d.crown_area_m2,d.iou,s=15,alpha=.45,label=site)
ax.set_xscale("log"); ax.set_ylim(-.02,1.02); ax.set_xlabel("Reference crown area (m², log scale)"); ax.set_ylabel("SAM3 IoU")
ax.set_title("SAM3 crown delineation vs crown size"); ax.legend(); fig.tight_layout()
fig.savefig(OUT/"sam3_iou_vs_crown_area.png",dpi=180); plt.show()

fig,ax=plt.subplots(figsize=(7,5))
data=[ok.loc[ok.site==s,"iou"].dropna().values for s in SITES]
ax.boxplot(data,tick_labels=list(SITES),showfliers=False); ax.set_ylim(0,1); ax.set_ylabel("IoU"); ax.set_title("SAM3 crown IoU by site")
fig.tight_layout(); fig.savefig(OUT/"sam3_iou_by_site.png",dpi=180); plt.show()


## 7 — Visual QA: best, median and worst cases per site

These panels are essential: a scalar IoU cannot distinguish sensible crown-boundary disagreement from catastrophic selection of an adjacent crown.


In [ ]:
def draw_case(site,cid,label):
    row=ANNS[site].loc[ANNS[site].crown_id==cid].iloc[0]
    with rasterio.open(SITES[site]["raster"]) as src:
        rgb,ref,box,tr=crop_for_crown(src,row.geometry); pred,score=sam3_box_predict(rgb,box)
    if pred.shape!=ref.shape:
        pred=np.array(Image.fromarray(pred.astype(np.uint8)).resize((ref.shape[1],ref.shape[0]),resample=Image.Resampling.NEAREST)).astype(bool)
    fig,ax=plt.subplots(figsize=(6,6)); ax.imshow(rgb)
    ax.contour(ref.astype(float),levels=[.5],linewidths=2)
    ax.contour(pred.astype(float),levels=[.5],linewidths=2,linestyles="--")
    x0,y0,x1,y1=box; ax.add_patch(plt.Rectangle((x0,y0),x1-x0,y1-y0,fill=False,linewidth=1.2))
    m=ok[(ok.site==site)&(ok.crown_id==cid)].iloc[0]
    ax.set_title(f"{site} — {label} — IoU={m.iou:.3f}, Dice={m.dice:.3f}"); ax.axis("off"); fig.tight_layout()
    fn=VIS/f"{site}_{label}_crown_{cid}.png"; fig.savefig(fn,dpi=180); plt.show()

for site,d in ok.groupby("site"):
    d=d.sort_values("iou")
    picks={"worst":int(d.iloc[0].crown_id),"median":int(d.iloc[len(d)//2].crown_id),"best":int(d.iloc[-1].crown_id)}
    for label,cid in picks.items(): draw_case(site,cid,label)


## 8 — Decision gate for NB06

This notebook does **not** define a universal pass/fail threshold in advance. We inspect:

- median and distribution of IoU/Dice by site;
- systematic area bias;
- centroid displacement;
- degradation for smaller crowns;
- visual failure modes, especially adjacent-crown capture, merged crowns and fragmented crowns.

If zero-shot SAM 3 delineation is sufficiently stable, NB06 can move to **automatic crown discovery / exemplar prompting** and then combine SAM-derived objects with KVO_04 Musanga probabilities. If it is weak, the benchmark still provides a useful negative result and tells us whether prompt refinement or domain-specific crown segmentation is required.

The output CSV preserves every crown-level result, so alternative quality thresholds can be explored without rerunning SAM 3.
